<h1 style="color: #ADD8E6;">Complementaria 7: Cadenas de Markov en Python I</h1>

<h2 style="color: #ADD8E6;">Introducción a jmarkov</h2>


En esta sección construiremos una cadena de Markov en tiempo continuo (CMTC) sencilla con 3 estados. La idea de este ejercicio es ilustrar, de manera compacta, los principales cálculos que podemos realizar con `jmarkov`, tanto estructurales como probabilísticos.

Consideremos una cadena con variable $X\left(t\right)$ y un espacio de estados $S_x = \{1,2,3\}$. Las transiciones entre estados están gobernadas por la siguiente matriz generadora:


$$
Q_{(i,j)} = \begin{pmatrix}
-3 & 2 & 1 \\
1 & -4 & 3 \\
2 & 1 & -3
\end{pmatrix}
$$


> **Recordemos que en una CMTC**
> - Las filas de $Q$ suman cero.
> - Los elementos fuera de la diagonal son no negativos.
> - La diagonal es negativa y representa la tasa de salida del estado.


In [47]:
import numpy as np
from jmarkov.ctmc import ctmc

# Definimos la matriz generadora
Q = np.array([
    [-3, 2, 1],
    [1, -4, 3],
    [2, 1, -3]
])

# Verificamos que las filas sumen cero
print("Suma por filas:", Q.sum(axis=1))

# Creamos la cadena de jmarkov continua
cadena = ctmc(Q)

Suma por filas: [0 0 0]


Podemos verificar propiedades importantes de la cadena.

In [48]:
print("Número de estados:", cadena.n_states)

print("¿Es irreducible?:", cadena.is_irreducible())

print("¿Es ergódica?:", cadena.is_ergodic())

Número de estados: 3
¿Es irreducible?: True
¿Es ergódica?: True



<h3 style="color: #ADD8E6;">Análisis transitorio</h3>

Podemos calcular la distribución en un instante $ t $ dado un estado inicial indicado con un vector de estados inciales $ \overrightarrow{\pi}(0) $.

$$
\overrightarrow{\pi}(t) = \overrightarrow{\pi}(0)e^{Qt},
$$

Supongamos que la cadena inicia en el estado 0 (esto lo indicamos con un 1 en el estado correspondiente):

$$
\overrightarrow{\pi}(t) = (1, 0, 0)
$$

`jmarkov` recibe como parámetros `t` y `alpha` para encontrar el vector de probabilidades $\overrightarrow{\pi}(t)$. Estos son el tiempo total que se esta evaluando y el vector de estados iniciales respectivamente. 

In [49]:
pi_0 = np.array([1, 0, 0])
t = 2

p_t = cadena.transient_probabilities(t=t, alpha=pi_0)

print(f"Distribución en t={t}:")
print(p_t)
print(" ")

# Asi por ejemplo, la probabilidad de estar en el estado 1 dado que empezamos en el estado 0,
#  luego de que transcurran t = 2 uniades de tiempo es:
print("La probabilidad de estar en el estado 1 en t=2 es", round(p_t[1], 3))

Distribución en t=2:
[0.34615261 0.26926285 0.38458454]
 
La probabilidad de estar en el estado 1 en t=2 es 0.269


<h3 style="color: #ADD8E6;">Probabilidades en estado estable</h3>

Si la cadena es ergódica, existe una distribución estacionaria $ \pi $ tal que:

$$
\overrightarrow{\pi}\mathbb{Q =}\overrightarrow{0},
$$

$$
\sum_{i \in S}^{}\pi_{i} = 1
$$



Podemos calcularla directamente:

In [50]:
pi = cadena.steady_state()

print("Distribución estacionaria:")
print(pi)
print(" ")

# Asi por ejemplo, la probabilidad de estar en el estado 1 en el largo plazo es:
print("La probabilidad de estar en el estado 1 en el largo plazo es", round(pi[1], 3))

Distribución estacionaria:
[0.34615385 0.26923077 0.38461538]
 
La probabilidad de estar en el estado 1 en el largo plazo es 0.269


<h3 style="color: #ADD8E6;">Tiempo de primera pasada</h3>

El tiempo promedio de primera pasada (hitting time) desde un estado inicial hacia un estado objetivo puede calcularse al escribir $m_{i,j}$

$$
m_{i,j} = \frac{1}{r_{ij}}\ \left( 1 + \sum_{k \neq j}^{}{q_{ik}m_{k,j}} \right)
$$

Esta expresión permite encontrar el tiempo $m_{i,j}$ a partir de los otros tiempos $m_{k,j}$, siendo $1/r_{i} = - 1/q_{ii}$. Matemáticamente, para encontrar $m_{i,j}$ hay que solucionar el sistema de ecuaciones simultaneas asociado. `jmarkov` ya implementa una función para encontrar los tiempos de primera pasada a un estado objetivo.

Por ejemplo, el tiempo promedio para llegar al estado 2 comenzando desde el estado 1 se calcula de la siguiente forma:

In [51]:
# Se define un array con los nombres de los estados
estados = ['0', '1', '2']
estados = np.array(estados)

estado_inicial = np.where(estados == '1')
estado_futuro = np.where(estados == '2')

# Calcular el tiempo
fpts = cadena.first_passage_time(estado_futuro)
# No tiene sentido calcular el tiempo de primera pasada al estado 2 desde el estado 2
# Por lo tanto, el arreglo que se regresa es de n-1 estados.
print("arreglo de tiempos de primera pasada al estado 2:")
print(fpts)
print(" ")

# La variable estado_inicial contiene el índice del estado '1', que es el estado desde el cual 
# queremos calcular el tiempo de primera pasada al estado '2'.
print("indice del estado inicial:", estado_inicial)
print(" ")

# Indexamos el tiempo de primera pasada desde el estado '1' al estado '2' utilizando el índice obtenido
# utilizamos [0][0] para obtener el numero directamente del array
tiempo_1_2 = fpts[estado_inicial][0][0]
print("El tiempo de primera pasada desde el estado 1 al estado 2 es:", round(tiempo_1_2, 3))

arreglo de tiempos de primera pasada al estado 2:
[[0.6]
 [0.4]]
 
indice del estado inicial: (array([1]),)
 
El tiempo de primera pasada desde el estado 1 al estado 2 es: 0.4


<h3 style="color: #ADD8E6;">Tiempo de ocupación</h3>

En una cadena de Markov en tiempo continuo, el **tiempo de ocupación** mide cuánto tiempo, en promedio, el proceso pasa en cada estado durante un intervalo finito $[0,t]$.

$$
\mathbf{M}^{t} = \int_{0}^{t}{e^{\mathbf{Q}u}du}
$$

En la matriz resultante:

- Cada fila corresponde al estado inicial.
- Cada columna corresponde al estado donde se acumula el tiempo.

Por lo tanto para nuestro ejemplo, la matriz devuelta por `occupation_time(T)` tiene la forma:

$$
\mathbf{M}^{t} =
\begin{pmatrix}
\text{Tiempo desde 0 en 0} & \text{Tiempo desde 0 en 1} & \text{Tiempo desde 0 en 2} \\
\text{Tiempo desde 1 en 0} & \text{Tiempo desde 1 en 1} & \text{Tiempo desde 1 en 2} \\
\text{Tiempo desde 2 en 0} & \text{Tiempo desde 2 en 1} & \text{Tiempo desde 2 en 2}
\end{pmatrix}
$$

In [52]:

tiempo_ocupacion = cadena.occupation_time(T=4)

print("Tiempo esperado en cada estado en 2 unidades de tiempo:")
print(tiempo_ocupacion)
print(" ")

sum_tiempo_ocupacion = tiempo_ocupacion.sum(axis=1)
print("Suma del tiempo de ocupación por filas:", sum_tiempo_ocupacion)
print(" ")

print("Tiempo en 2 desde 1:", round(tiempo_ocupacion[1][2], 3))

Tiempo esperado en cada estado en 2 unidades de tiempo:
[[1.52070832 1.0502945  1.42899215]
 [1.28993909 1.20414066 1.50591523]
 [1.32840063 1.01183297 1.65976138]]
 
Suma del tiempo de ocupación por filas: [3.99999497 3.99999497 3.99999497]
 
Tiempo en 2 desde 1: 1.506


Ahora que vimos como utilizar las funciones básicas de `jmarkov` en cadenas continuas sigamos con un ejercicio más elaborado.

<h2 style="color: #ADD8E6;">Ejercicio de implementación: Hospital en tiempo continuo</h2>

Considere un hospital con $B = 20$ camas. Hay dos tipos de pacientes que usan este hospital. Los pacientes de tipo $i\ (i=1,2)$ llegan según un proceso de Poisson con tasa $\lambda_1 = 4$ y $\lambda_2 = 3$ pacientes por hora y requieren un servicio independiente e idénticamente distribuido (iid) con tiempos de servicio que siguen una distribución exponencial con parámetros $\mu_1 = 2$ y $\mu_2 = 1.5$ pacientes por hora. El hospital utiliza la siguiente política: reserva $b = 5$ camas para pacientes de tipo 1. Así, cuando un paciente de tipo 1 llega, se le admite en el hospital siempre que haya una cama disponible. Sin embargo, un paciente de tipo 2 es admitido solo si el número de pacientes de tipo 2 que actualmente ocupan camas es menor que $B-b$.

Se le ha solicitado modelar la anterior situación definiendo claramente variables de estado, espacios de estado y matriz de transición o generadora según sea el caso.


<h3 style="color: #ADD8E6;">Preguntas</h3>

1. Construya la matriz de tasas de transición $\mathbb{Q}$, luego cree la cadena de Markov utilizando la librería `jmarkov`.
2. Calcule la probabilidad de que después de 6 horas el hospital tenga 1 paciente tipo 1 y 3 pacientes tipo 2, dado que al inicio está vacío. Utilice `jmarkov` y compare con el resultado por simulación.

<h4 style="color: #ADD8E6;">1. Construir la matriz de de transición y la cadena en jmarkov</h4>

**Variables de estado:**
- $X\left(t\right)$: El número de pacientes tipo 1 en el tiempo $t$.
- $Y\left(t\right)$: El número de pacientes tipo 2 en el tiempo $t$.
- $Z\left(t\right)$: $\{X\left(t\right),\ Y\left(t\right)\}$.

**Espacio de estados:**

$$
S_X=\{0,\ 1,\ 2,\ 3,\ \ldots,B\}
$$

$$
S_Y=\{0,\ 1,\ 2,\ 3,\ \ldots,B-b\}
$$

$$
S_Z=\{(i,j):0\le i\le B,\ 0\le j\le B-b\}
$$

**Matriz generadora:**

$$
Q((i,j),(i',j')) = \begin{cases}
\lambda_1, & \text{si } i' = i + 1,\; j' = j,\; i < B, \\[1mm]
\lambda_2, & \text{si } j' = j + 1,\; i' = i,\; j < B - b, \\[1mm]
i \cdot \mu_1, & \text{si } i' = i - 1,\; j' = j,\; i > 0, \\[1mm]
j \cdot \mu_2, & \text{si } j' = j - 1,\; i' = i,\; j > 0, \\[1mm]
0, & \text{en otro caso (d.l.c.)}.
\end{cases}
$$

A continuación, se implementa el modelo en Python.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from jmarkov.ctmc import ctmc

# Se definen los parámetros
B = 20  # Número total de camas
b = 5   # Número de camas reservadas para pacientes tipo 1
lambda_1 = 4  # Tasa de llegada de pacientes tipo 1 (por hora)
lambda_2 = 3  # Tasa de llegada de pacientes tipo 2 (por hora)
mu_1 = 2      # Tasa de salida de pacientes tipo 1 (por hora)
mu_2 = 1.5    # Tasa de salida de pacientes tipo 2 (por hora)

# Dimensión de la matriz Q
num_estados = (B + 1) * (B - b + 1)

# Inicializamos la matriz Q con ceros
Q_evaluar = np.zeros((num_estados, num_estados), dtype=float)

# Función para mapear (i, j) a índice en la matriz Q
def index(i, j):
    return i * (B - b + 1) + j

# Llenar la matriz Q
for i in range(B + 1):
    for j in range(B - b + 1):
        idx = index(i, j)

        # Llegada de un paciente tipo 1
        if i < B:
            Q_evaluar[idx, index(i + 1, j)] = lambda_1

        # Llegada de un paciente tipo 2 (si hay espacio disponible)
        if j < B - b:
            Q_evaluar[idx, index(i, j + 1)] = lambda_2

        # Salida de un paciente tipo 1
        if i > 0:
            Q_evaluar[idx, index(i - 1, j)] = i * mu_1

        # Salida de un paciente tipo 2
        if j > 0:
            Q_evaluar[idx, index(i, j - 1)] = j * mu_2

# Llenar la diagonal con la suma negativa de las filas de la matriz
for i in range(num_estados):
    Q_evaluar[i, i] = -np.sum(Q_evaluar[i, :])
    
print(Q_evaluar)

[[ -7.    3.    0.  ...   0.    0.    0. ]
 [  1.5  -8.5   3.  ...   0.    0.    0. ]
 [  0.    3.  -10.  ...   0.    0.    0. ]
 ...
 [  0.    0.    0.  ... -62.5   3.    0. ]
 [  0.    0.    0.  ...  21.  -64.    3. ]
 [  0.    0.    0.  ...   0.   22.5 -62.5]]


Para crear la cadena de markov utilizaremos la instancia `ctmc` la libreria `jmarkov`, ya que estamos creando una cadena de tiempo discreto.

In [54]:
from jmarkov.ctmc import ctmc

# Crear la cadena de Markov en tiempo continuo usando jmarkov
hospital_chain = ctmc(Q_evaluar)

<h4 style="color: #ADD8E6;">2. Cálculo de la probabilidad de que, después de 6 horas, el hospital tenga 1 pacientes tipo 1 y 3 pacientes tipo 2</h4>

<h5 style="color: #ADD8E6;">Método en jmarkov</h5>

Para resolver a esta pregunta primero se debe establecer correctamente el vector de estado inicial (hospital vacío).

In [ ]:
# --------------------------------------------------------------------
# Pregunta 2: Probabilidad de que después de 6 horas el hospital tenga 1 paciente tipo 1, y 3
# pacientes tipo 2, dado que al inicio está vacío.
alpha = np.zeros(num_estados)
alpha[index(0, 0)] = 1

Una vez definido este vector, se puede calcular la distribución de probabilidad a las 6 horas utilizando la función `transient_probabilities` de `jmarkov`

In [56]:
# Probabilidades transitorias en 6 horas
probs_transitorias_6h = hospital_chain.transient_probabilities(t=6, alpha=alpha)

Con esto podemos identificar que la probabilidad correspondiente al estado $(1,3)$ es la siguiente:

In [57]:
# Probabilidad de que el estado sea (1,3)
prob_1_3 = probs_transitorias_6h[index(1, 3)]
print(f"Probabilidad de que después de 6 horas haya 1 pacientes tipo 1 y 3 tipo 2: {prob_1_3:.6f}")

Probabilidad de que después de 6 horas haya 1 pacientes tipo 1 y 3 tipo 2: 0.048836


<h5 style="color: #ADD8E6;">Comparación con Simulación en SimPy</h5>

En las sesiones anteriores utilizamos **SimPy** para simular sistemas de colas. Dado que una Cadena de Markov en Tiempo Continuo (CMTC) se basa en eventos que ocurren en el tiempo (llegadas, salidas, transiciones), podemos usar el entorno de SimPy para simular el comportamiento dinámico de nuestro hospital.

Para esto, simularemos el hospital múltiples veces, parando el reloj de SimPy exactamente a las 6 horas, para revisar en qué estado se encuentra.

> **¿Cómo modelamos una matriz $Q$ en SimPy?**
> 1. Calculamos la tasa de salida ($-q_{ii}$) usando la diagonal de la matriz.
> 2. Le pedimos a SimPy que "espere" por un tiempo aleatorio exponencial: `yield env.timeout(tiempo_permanencia)`.
> 3. Una vez transcurrido el tiempo, calculamos las probabilidades de salto de la cadena $p_{ij} = q_{ij} / -q_{ii}$ y hacemos la transición a un nuevo estado.

In [63]:
import simpy
import numpy as np

# Parámetros de la simulación
n_simulaciones = 10000    
tiempo_evaluacion = 6.0   
estado_inicial_idx = index(0, 0)
estado_objetivo = index(1, 3) 


# Proceso para simular los saltos de la CMTC
def proceso_markov_simpy(env, Q_matriz, est_inicial, estado_monitor):

    estado_actual = est_inicial
    n_estados = Q_matriz.shape[0]
    
    while True:

        # Se guarda el estado actual en el monitor
        estado_monitor['actual'] = estado_actual
        
        # Leer la tasa de salida de la matriz Q
        tasa_salida = -Q_matriz[estado_actual, estado_actual]

        # Simular tiempo de permanencia y avanzar el reloj
        tiempo_permanencia = np.random.exponential(scale = 1/tasa_salida)
        yield env.timeout(tiempo_permanencia)
        
        # Transición de estado
        probabilidades_salto = Q_matriz[estado_actual, :].copy()
        probabilidades_salto[estado_actual] = 0.0
        probabilidades_salto = probabilidades_salto / tasa_salida
        
        # Se selecciona el nuevo estado
        estado_actual = np.random.choice(n_estados, p=probabilidades_salto)

# Ejecutar Montecarlo
np.random.seed(0)
casos_favorables = 0

for i in range(n_simulaciones):
    # Crear un nuevo entorno para cada réplica
    env = simpy.Environment()
    
    # Se define un diccionario para recuperar el valor de la función
    monitor = {'actual': estado_inicial_idx} 
    
    # Inicia el proceso
    env.process(proceso_markov_simpy(env, Q_evaluar, estado_inicial_idx, monitor))
    
    # Correr la simulación por 6 horas
    env.run(until=tiempo_evaluacion)
    
    # Revisar si al final de las 6 horas el sistema se quedó en el estado (1,3)
    if monitor['actual'] == estado_objetivo:
        casos_favorables += 1

# Calcular la probabilidad estimada
prob_estimada = casos_favorables / n_simulaciones

print(f"Probabilidad teórica (jmarkov): {prob_1_3:.6f}")
print(f"Probabilidad estimada (SimPy con {n_simulaciones} réplicas): {prob_estimada:.6f}")

error_relativo = abs(prob_estimada - prob_1_3) / prob_1_3 * 100
print(f"Error relativo: {error_relativo:.2f}%")

Probabilidad teórica (jmarkov): 0.048836
Probabilidad estimada (SimPy con 10000 réplicas): 0.049700
Error relativo: 1.77%


Al igual que en casos anteriores, podemos observar cómo la simulación logra una excelente aproximación al resultado exacto resuelto mediante la matriz exponencial de la distribución transitoria en `jmarkov`. 

Por el teorema de la Ley de los números grandes, a medida que aumentemos el número de réplicas (`n_simulaciones`), nuestra probabilidad estimada convergerá de manera mucho más exacta a la probabilidad teórica calculada por la herramienta analítica.

Universidad de los Andes | Vigilada Mineducación. Reconocimiento como Universidad: Decreto 1297 del 30 de mayo de 1964. Reconocimiento personería jurídica: Resolución 28 del 23 de febrero de 1949 Minjusticia. Departamento de Ingeniería Industrial Carrera 1 Este No. 19 A 40 Bogotá, Colombia Tel. (57.1) 3324320 | (57.1) 3394949 Ext. 2880 /2881 http://industrial.uniandes.edu.co